In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv("BRFSS_2025_NC_Final_Analysis.csv")

In [3]:
age_map = {
    1: "18-24", 2: "25-34", 3: "25-34",
    4: "35-44", 5: "35-44",
    6: "45-54", 7: "45-54",
    8: "55-64", 9: "55-64",
    10: "65-74", 11: "65-74",
    12: "75+", 13: "75+"
}
df["Age_Category"] = df["Age_Group"].map(age_map)

df["Smoking_Category"] = df["Smoking_Status"].replace({
    "Never": "Never smoker",
    "Former": "Former smoker",
    "Current daily": "Current smoker",
    "Current some days": "Current smoker"
})

income_map = {
    "1": "Less than $15,000",
    "2": "15,000-24,999",
    "3": "25,000-34,999",
    "4": "35,000-49,999",
    "5": "50,000-99,999",
    "6": "$100,000 or more",
    "7": "$100,000 or more",
    "Missing/unknown": "Unknown / missing"
}
df["Income_Category"] = (
    df["Income_Group"].astype(str).map(income_map)
    .fillna("Unknown / missing")
)

groups = {
    "Overall sample": df,
    "No diabetes": df[df["Diabetes_Binary"] == 0],
    "Diagnosed diabetes": df[df["Diabetes_Binary"] == 1]
}

rows = []

def add_header(label):
    rows.append([label, "", "", ""])

def add_categorical(column, categories):
    for category in categories:
        row = [category]
        for group in groups.values():
            n = (group[column] == category).sum()
            valid = group[column].notna().sum()
            pct = 100 * n / valid if valid else np.nan
            row.append(f"{n:,} ({pct:.1f}%)")
        rows.append(row)

def add_continuous(column):
    mean_row = ["Mean (SD)"]
    median_row = ["Median (IQR)"]

    for group in groups.values():
        x = pd.to_numeric(
            group[column], errors="coerce"
        ).dropna()

        mean_row.append(
            f"{x.mean():.2f} ({x.std():.2f})"
        )

        median_row.append(
            f"{x.median():.2f} "
            f"({x.quantile(.25):.2f}–{x.quantile(.75):.2f})"
        )

    rows.append(mean_row)
    rows.append(median_row)


rows.append([
    "Sample size, n",
    *[f"{len(g):,}" for g in groups.values()]
])

add_header("DEMOGRAPHIC CHARACTERISTICS")

add_header("Age group, n (%)")
add_categorical("Age_Category", [
    "18-24", "25-34", "35-44", "45-54",
    "55-64", "65-74", "75+"
])

add_header("Sex, n (%)")
add_categorical("Sex", ["Male", "Female"])

add_header("Annual household income, n (%)")
add_categorical("Income_Category", [
    "Less than $15,000",
    "15,000-24,999",
    "25,000-34,999",
    "35,000-49,999",
    "50,000-99,999",
    "$100,000 or more",
    "Unknown / missing"
])

add_header("HEALTH INDICATORS")

add_header("BMI (kg/m²)")
add_continuous("BMI")

add_header("LIFESTYLE CHARACTERISTICS")

add_header("Physical activity, n (%)")
add_categorical("Physical_Activity", ["Yes", "No"])

add_header("Smoking status, n (%)")
add_categorical("Smoking_Category", [
    "Never smoker",
    "Former smoker",
    "Current smoker"
])

add_header("Fruit consumption (times/day)")
add_continuous("Fruit_Times_Per_Day")

add_header("Vegetable consumption (times/day)")
add_continuous("Vegetable_Times_Per_Day")

# Create and display Table 1
table1 = pd.DataFrame(
    rows,
    columns=[
        "Characteristic",
        "Overall sample",
        "No diabetes",
        "Diagnosed diabetes"
    ]
)

print("Table 1: Characteristics of the NC Sample")
display(table1.set_index("Characteristic"))


Table 1: Characteristics of the NC Sample


,Overall sample,No diabetes,Diagnosed diabetes
Characteristic,,,
"Sample size, n","2,682","2,258",424
DEMOGRAPHIC CHARACTERISTICS,,,
"Age group, n (%)",,,
18-24,233 (8.7%),231 (10.2%),2 (0.5%)
25-34,355 (13.2%),345 (15.3%),10 (2.4%)
35-44,375 (14.0%),356 (15.8%),19 (4.5%)
45-54,370 (13.8%),311 (13.8%),59 (13.9%)
55-64,408 (15.2%),322 (14.3%),86 (20.3%)
65-74,524 (19.5%),394 (17.4%),130 (30.7%)
